# 05 — Reinforcement Learning: PPO Training
**Goal**: Train the SFT model with PPO and sandbox execution rewards. Two paper arms:

| Arm | `reward_mode` | Checkpoint |
|---|---|---|
| RQ3 dense | `dense` (partial credit) | `./checkpoints/ppo_dense/final` |
| RQ4 binary | `binary` (AC→1 else 0) | `./checkpoints/ppo_binary/final` |

Shared HPs: frozen SFT ref (`init_kl_coef=0.05`, `target_kl=6.0`, `kl_penalty=abs`, LR `1e-6`), fixed `seed`, packed multi-test partial credit for dense mode.

**Paper runs**: set `MAX_STEPS=100` (or higher) below. `MAX_STEPS=10` is diagnostic only.

---


## Step 1: Environment Setup & Checkpoint Resolution

In [1]:
!pip install -q "trl<0.12.0" peft transformers datasets bitsandbytes accelerate

import sys, os, shutil

# 1. Pull latest code from junior-A branch
repo_root = os.path.abspath(os.getcwd())
if os.path.isdir('/kaggle/working'):
    !rm -rf /kaggle/working/src
    !git clone -b junior-A https://github.com/Oin19/self-correction-llm-rl.git temp_repo
    !cp -r temp_repo/src /kaggle/working/src
    GIT_SHA = !git -C temp_repo rev-parse HEAD
    GIT_SHA = GIT_SHA[0] if GIT_SHA else ''
    print(f'Cloned junior-A @ {GIT_SHA}')
    !rm -rf temp_repo
    repo_root = '/kaggle/working'

# 2. Path setup (expander/KL fixes now live in repo src/training/ppo.py)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path initialized: {repo_root}")

# 3. Uninstall torchao to prevent PEFT version conflicts
!pip uninstall -y torchao

# 4. Resolve & Copy SFT Checkpoint
SFT_PATH = './checkpoints/sft/final'
if not os.path.exists(SFT_PATH) and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files and 'sft' in root.lower():
            dest = '/kaggle/working/checkpoints/sft/final'
            os.makedirs(dest, exist_ok=True)
            for item in os.listdir(root):
                s = os.path.join(root, item)
                d = os.path.join(dest, item)
                if os.path.isfile(s): shutil.copy2(s, d)
                elif os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
            SFT_PATH = dest
            print(f"SUCCESS: SFT checkpoint copied to '{dest}'!")
            break

print(f"SFT Checkpoint ready at: {SFT_PATH}")

if 'GIT_SHA' not in dir():
    GIT_SHA = ''


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 16.0 MB/s eta 0:00:00
Cloning into 'temp_repo'...
remote: Enumerating objects: 884, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (273/273), done.
remote: Total 884 (delta 215), reused 167 (delta 76), pack-reused 522 (from 1)
Receiving objects: 100% (884/884), 457.08 KiB | 7.25 MiB/s, done.
Resolving deltas: 100% (502/502), done.
Cloned junior-A @ 36c8055b19891eb279317d187acb5cc53e1848e5
Project path initialized: /kaggle/working
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
SUCCESS: SFT checkpoint copied to '/kaggle/working/checkpoints/sft/final'!
SFT Checkpoint ready at: /kaggle/working/checkpoints/sft/final


## Step 2: Load APPS Dataset & Tokenizer

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from src.training.ppo import run_ppo_training, normalize_tests

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading APPS training dataset for PPO (1,000 problem pairs)...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:1000]')
# load_from_cache_file=False: stale HF cache can yield empty apps_clean after normalize_tests changes
apps_clean = apps.filter(lambda x: len(normalize_tests(x)) > 0, load_from_cache_file=False)
if len(apps_clean) == 0:
    sample = apps[0]
    raise ValueError(f'Cell4 filter kept 0/{len(apps)} rows; columns={sorted(sample.keys())}')
print(f"Prepared {len(apps_clean)} APPS problems with valid benchmark tests for PPO rollout.")

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading APPS training dataset for PPO (1,000 problem pairs)...


all/train/0000.parquet:   0%|          | 0.00/45.0M [00:00<?, ?B/s]

competition/train/0000.parquet:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

... (7 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Prepared 1000 APPS problems with valid benchmark tests for PPO rollout.


## Step 3: Pre-Flight Reward Unit Test (dense + binary)

In [3]:
from src.execution.executor import PythonSandbox
from src.rewards.execution_reward import score_rollout_reward

sandbox = PythonSandbox(default_timeout=5.0, max_memory_mb=1024.0)
tests = [{"assertion": "assert add(1,2) == 3"}] * 5 + [{"assertion": "assert add(2,3) == 5"}] * 5
cases = {
    "perfect": "def add(a,b): return a+b",
    "partial": "def add(a,b): return a+b if a==1 else 0",
    "bad": "def add(a,b): return a+",
}
expected_dense = {"perfect": 1.0, "partial": 0.5, "bad": -0.2}
expected_binary = {"perfect": 1.0, "partial": 0.0, "bad": 0.0}
for name, code in cases.items():
    result = sandbox.run_tests(code, tests)
    d = score_rollout_reward(result, reward_mode="dense")
    b = score_rollout_reward(result, reward_mode="binary")
    print(f"{name}: status={result.status} passed={result.passed_tests}/{result.total_tests} dense={d:.3f} binary={b:.3f}")
    assert abs(d - expected_dense[name]) < 1e-9, name
    assert abs(b - expected_binary[name]) < 1e-9, name
print("PASS: dense partial + binary AC-only preflight verified.")


perfect: status=ExecutionStatus.AC passed=10/10 dense=1.000 binary=1.000
partial: status=WA passed=5/10 dense=0.500 binary=0.000
bad: status=ExecutionStatus.CE passed=0/10 dense=-0.200 binary=0.000
PASS: dense partial + binary AC-only preflight verified.


## Step 4: Run PPO Training (dense for RQ3 / binary for RQ4)

Flip `REWARD_MODE` and re-run for the second arm. Paper default `MAX_STEPS=100`.

In [4]:
# Paper arms: REWARD_MODE = "dense" -> ./checkpoints/ppo_dense
#              REWARD_MODE = "binary" -> ./checkpoints/ppo_binary
REWARD_MODE = "binary"   # or "binary"
MAX_STEPS = 100         # use 10 only for smoke/diagnostic
SEED = 42
OUTPUT_DIR = f"./checkpoints/ppo_{REWARD_MODE}"

print(f"=== PPO Training | mode={REWARD_MODE} steps={MAX_STEPS} seed={SEED} -> {OUTPUT_DIR} ===")

ppo_trainer = run_ppo_training(
    sft_model_path=SFT_PATH,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir=OUTPUT_DIR,
    num_epochs=1,
    learning_rate=2e-7,
    batch_size=2,
    mini_batch_size=1,
    gradient_accumulation_steps=2,
    init_kl_coef=0.30,
    target_kl=4.0,
    max_steps=MAX_STEPS,
    reward_mode=REWARD_MODE,
    seed=SEED,
    commit_sha=GIT_SHA if 'GIT_SHA' in dir() else '',
)

print(f"\nPPO Training completed! Saved to {OUTPUT_DIR}/final")


=== PPO Training | mode=binary steps=100 seed=42 -> ./checkpoints/ppo_binary ===
PPO seed=42 reward_mode=binary output_dir=./checkpoints/ppo_binary commit=36c8055b19891eb279317d187acb5cc53e1848e5
Benchmark tests: n=1000 min=1 max=1 mean=1.00 multi_case=0/1000 packed_multi=448/1000 packed_total_max=123
  test_src[0]: normalize=1 io_inputs=1 test_list=0 test_str_len=0 packed_tests=4
  test_src[1]: normalize=1 io_inputs=1 test_list=0 test_str_len=0 packed_tests=3
  test_src[2]: normalize=1 io_inputs=1 test_list=0 test_str_len=0 packed_tests=5


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

... (2 outputs left out)

/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(


PPO sample step=1 idx=0: status=ExecutionStatus.CE passed=0/1 reward=0.000
Generated code preview: : int, s: int) -> int:         if s & (1 << u):             return 0         if dp[u]:             return dp[u]         if s + 1 == (1 << (u + 1)):             return dp[u] = 1 if keys[u + 1] == -1 else 0         c = count[A[u]]         result += f(u + 1, s | 1 << u)         count[A[u]] = c - 1 if c > 1 else 0         for v in range(u + 1, len(A)):             if count[A[v]] > 0 and keys[v] != -1 and keys[u] > keys[v]:                 if (1 << (v - u - 1)) & s:                     continue                 if (1 << (keys[u] - keys[v])) & s:                     continue                 if ((A[u] << 2) | A[v]) != ((A[u] << 2) | A[v]).bit_length():                     continue                 count[A[v]] -= 1                 dp[v] += f(v, s | 1 << v)         count[A[u]] = c         if count[A[
PPO sample step=1 idx=1: status=ExecutionStatus.AC passed=1/1 reward=1.000
Generated code preview: d

## Step 5: Checkpoint Verification & Reload Test

In [5]:
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

PPO_PATH = f"./checkpoints/ppo_{REWARD_MODE}/final"
print(f"=== Checkpoint Reload Verification: {PPO_PATH} ===")
if os.path.isfile(os.path.join(PPO_PATH, "adapter_config.json")):
    import json as _json
    meta_path = os.path.join(PPO_PATH, "ppo_metadata.json")
    if os.path.isfile(meta_path):
        with open(meta_path, encoding="utf-8") as f:
            print("metadata:", _json.load(f))
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True
    )
    ppo_model = PeftModel.from_pretrained(base_model, PPO_PATH)
    print("SUCCESS: Reloaded PPO model successfully into PeftModel!")
else:
    raise FileNotFoundError(f"PPO checkpoint not found at {PPO_PATH}")


=== Checkpoint Reload Verification: ./checkpoints/ppo_binary/final ===
metadata: {'base_model': 'deepseek-ai/deepseek-coder-1.3b-instruct', 'checkpoint_type': 'ppo_policy_adapter', 'reward_mode': 'binary', 'reward_type': 'execution_binary_ac_only', 'execution_tests_required': True, 'seed': 42, 'commit_sha': '36c8055b19891eb279317d187acb5cc53e1848e5', 'max_steps': 100, 'learning_rate': 2e-07, 'init_kl_coef': 0.3, 'target_kl': 4.0, 'batch_size': 2, 'trainable_parameters': 3147777, 'total_parameters': 1349619713}


Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

SUCCESS: Reloaded PPO model successfully into PeftModel!
